# Deploying NVIDIA Nemotron-3-Super with TensorRT LLM

This notebook walks through running the `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B` model via TensorRT-LLM.

[TensorRT LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs.

**Base model:** `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16`

**Prerequisites:**
- NVIDIA GPU with >=264 GB VRAM for BF16 (4x H100), >=160 GB for FP8 (2x H100)
- CUDA 12.x
- Python 3.10+
- TensorRT-LLM container from [NGC](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release?version=1.3.0rc4)

## Container Setup

Run the following in a **host terminal**:

```shell
export CACHE_ROOT=/ephemeral
mkdir -p "$CACHE_ROOT/trtllm_cache"
mkdir -p "$CACHE_ROOT/trtllm_tmp"

docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all \
  -p 8000:8000 \
  -v "$CACHE_ROOT":"$CACHE_ROOT" \
  -e HF_HOME="$CACHE_ROOT/trtllm_cache" \
  -e HUGGINGFACE_HUB_CACHE="$CACHE_ROOT/trtllm_cache" \
  -e TMPDIR="$CACHE_ROOT/trtllm_tmp" \
  -e TEMP="$CACHE_ROOT/trtllm_tmp" \
  -e TMP="$CACHE_ROOT/trtllm_tmp" \
  nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc4
```

In [ ]:
# If pip not found
!python -m ensurepip --default-pip

In [ ]:
%pip install torch==2.9.1 openai==2.6.1 requests

## Verify GPU

In [ ]:
import sys
import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

## Start TensorRT-LLM Server

Run in the docker terminal in this same container:

### Create extra config
```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  enable_block_reuse: false
moe_config:
   backend: TRTLLM
cuda_graph_config:
    enable_padding: true
    batch_sizes: [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
EOF
```

### BF16 (4x H100)
```shell
mpirun -n 1 --allow-run-as-root --oversubscribe \
trtllm-serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --host 0.0.0.0 \
  --port 8000 \
  --backend pytorch \
  --max_batch_size 128 \
  --tp_size 4 --ep_size 4 \
  --max_num_tokens 16384 \
  --trust_remote_code \
  --reasoning_parser nano-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

### FP8 (2x H100)
```shell
mpirun -n 1 --allow-run-as-root --oversubscribe \
trtllm-serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8 \
  --host 0.0.0.0 \
  --port 8000 \
  --backend pytorch \
  --max_batch_size 128 \
  --tp_size 2 --ep_size 2 \
  --max_num_tokens 16384 \
  --trust_remote_code \
  --reasoning_parser nano-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

## 3. OpenAI-Compatible API Usage

In [ ]:
from openai import OpenAI
import requests

BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"  # BF16
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"  # FP8

In [ ]:
# Reasoning ON (default)
print("=== Reasoning ON ===")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about NVIDIA GPUs."}
    ],
    temperature=1,
    max_tokens=1024,
)
print("Reasoning:", response.choices[0].message.reasoning_content)
print("\nContent:", response.choices[0].message.content)

In [ ]:
# Reasoning OFF
print("=== Reasoning OFF ===")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(response.choices[0].message.content)

In [ ]:
# Streaming
print("=== Streaming response ===")
stream = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

In [ ]:
# Tool calling
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {"type": "integer", "description": "The total amount of the bill"},
                    "tip_percentage": {"type": "integer", "description": "The percentage of tip"}
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print("Reasoning:", completion.choices[0].message.reasoning_content)
print("Tool calls:", completion.choices[0].message.tool_calls)

## 4. Controlling Reasoning Budget

The `reasoning_budget` parameter limits the model's reasoning trace length.

In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer

class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. First call to get reasoning content
        response = self.client.chat.completions.create(
            model=model, messages=messages, max_tokens=reasoning_budget, **kwargs
        )
        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        assert remaining_tokens > 0, f"remaining tokens must be positive. Increase max_tokens or lower reasoning_budget."

        # 2. Append reasoning and get final answer
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, continue_final_message=True,
        )
        response = self.client.completions.create(
            model=model, prompt=prompt, max_tokens=remaining_tokens, **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [ ]:
# Test with bounded reasoning
budget_client = ThinkingBudgetClient(
    base_url="http://0.0.0.0:8000/v1",
    api_key="null",
    tokenizer_name_or_path=model_id
)

resp = budget_client.chat_completion(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=512,
    reasoning_budget=128
)

print("Reasoning:", resp["reasoning_content"], "\n\nContent:", resp["content"])

## 5. Inference with Fine-Tuned LoRA Adapter

Load the fine-tuned adapter from GCS and run inference.

In [ ]:
# To load a fine-tuned LoRA adapter from GCS and run inference
from google.cloud import storage
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import tempfile

BUCKET_NAME = "x402-477302-train2earn-datasets"
ADAPTER_PREFIX = "adapters/nemotron-clawd-finetune/"

def download_adapter_from_gcs(bucket_name, prefix, local_dir):
    """Download LoRA adapter from GCS."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)
    for blob in blobs:
        rel_path = blob.name[len(prefix):]
        dest = os.path.join(local_dir, rel_path)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        blob.download_to_filename(dest)
        print(f"  Downloaded: {rel_path}")

# Download adapter
with tempfile.TemporaryDirectory() as tmpdir:
    print("Downloading adapter from GCS...")
    download_adapter_from_gcs(BUCKET_NAME, ADAPTER_PREFIX, tmpdir)
    
    # Load base model
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto",
    )
    
    # Load and merge adapter
    model = PeftModel.from_pretrained(base_model, tmpdir)
    model = model.merge_and_unload()
    
    # Clawd agent inference
    messages = [
        {"role": "system", "content": "You are a Clawd agent specialized in Solana DeFi."},
        {"role": "user", "content": "Analyze the Phoenix perpetuals orderbook for SOL-PERP and suggest a trading strategy."}
    ]
    
    tokenized = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    
    outputs = model.generate(
        tokenized,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
    )
    
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Cleanup

1. Press `Ctrl+C` in the terminal running `trtllm-serve` to stop the server.
2. Run `exit` in the Docker shell (`--rm` removes it).
3. Optionally clear notebook-side CUDA cache:

In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")